# Repair Author Sequence Bug

One-time data repair for the author sequence mismatch in `work_authors`.

## Problem
`CreateWorkAuthors` joined `openalex_works_base` author positions to `works_legacy.work_authors` on `(work_id, author_sequence)`, but provenance-based ordering in `openalex_works_base` can differ from the legacy POSEXPLODE ordering. This misaligned ~8M author_ids across ~834K works. Additionally, poisoned `longest_name` values in `openalex_authors` caused `MatchAuthors` to assign incoming works to wrong entities, producing ~2.2M additional wrong matches.

## Phases

### Phase 2 (Cells 1-7): Legacy-based repair
- Scope: 834K works where legacy `work_authors` provides a concrete correction
- Method: Re-match to legacy by name (exact + parsed fallback), replace wrong author_ids
- Status: DONE

### Phase 4 (Cells 8-12): Broad mismatch repair
- Scope: ~2.2M additional mismatches detected by SOUNDEX + Levenshtein + script filters (~92% TP rate)
- Method: NULL wrong author_ids so `MatchAuthors` can re-match after `CreateAuthors` cleans profiles
- Prerequisite: Run `CreateAuthors` first to clean poisoned `longest_name` values

## Prerequisites
- Run on X-Large SQL warehouse (`3996dc0a9b183ce3`)
- Code fixes already deployed: block_key (b4748dc), COALESCE (07931d2)
- `CreateParsedNamesLookup` must be run first to ensure all names are in author_names (63d8456)

### Cell 1: Diagnostic — Quantify affected rows

Count work_authors rows where the parsed last name mismatch exists AND a legacy join (exact name OR parsed name fallback) gives a different author_id.

In [0]:
WITH display_mismatch AS (
    SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
           pn_raw.parsed_name.last AS raw_last, pn_raw.parsed_name.first AS raw_first
    FROM openalex.works.work_authors wa
    INNER JOIN openalex.authors.openalex_authors oa ON wa.author_id = oa.id
    INNER JOIN openalex.authors.author_names pn_raw ON TRIM(wa.raw_author_name) = pn_raw.raw_author_name
    INNER JOIN openalex.authors.author_names pn_display ON TRIM(oa.display_name) = pn_display.raw_author_name
    WHERE wa.work_id <= 7000000000
      AND wa.author_id IS NOT NULL
      AND pn_raw.parsed_name.last IS NOT NULL AND pn_raw.parsed_name.last != ''
      AND pn_display.parsed_name.last IS NOT NULL AND pn_display.parsed_name.last != ''
      AND pn_raw.parsed_name.last != pn_display.parsed_name.last
      AND pn_display.parsed_name.last NOT IN (pn_raw.parsed_name.first, pn_raw.parsed_name.middle, pn_raw.parsed_name.last)
      AND pn_raw.parsed_name.last NOT IN (pn_display.parsed_name.first, pn_display.parsed_name.middle, pn_display.parsed_name.last)
      AND EXISTS (
          SELECT 1 FROM openalex.works.work_authors wa2
          WHERE wa2.work_id = wa.work_id AND wa2.author_sequence > 0
      )
),
-- Legacy enriched with parsed names
legacy_parsed AS (
    SELECT lwa.work_id, LOWER(TRIM(lwa.raw_author_name)) AS name_norm, lwa.author_id, lwa.author_sequence,
           pn.parsed_name.last AS legacy_last, pn.parsed_name.first AS legacy_first,
           ROW_NUMBER() OVER (PARTITION BY lwa.work_id, LOWER(TRIM(lwa.raw_author_name)) ORDER BY lwa.author_sequence) AS rn_exact,
           ROW_NUMBER() OVER (PARTITION BY lwa.work_id, pn.parsed_name.last, pn.parsed_name.first ORDER BY lwa.author_sequence) AS rn_parsed
    FROM openalex.works_legacy.work_authors lwa
    INNER JOIN openalex.authors.author_names pn ON TRIM(lwa.raw_author_name) = pn.raw_author_name
    WHERE lwa.author_id IS NOT NULL
),
-- Pass 1: exact name match
exact_match AS (
    SELECT dm.work_id, dm.author_sequence, dm.author_id, lp.author_id AS legacy_author_id
    FROM display_mismatch dm
    INNER JOIN legacy_parsed lp
        ON dm.work_id = lp.work_id
        AND LOWER(TRIM(dm.raw_author_name)) = lp.name_norm
        AND lp.rn_exact = 1
    WHERE dm.author_id != lp.author_id
),
-- Pass 2: parsed name fallback for rows not matched in Pass 1
parsed_match AS (
    SELECT dm.work_id, dm.author_sequence, dm.author_id, lp.author_id AS legacy_author_id
    FROM display_mismatch dm
    INNER JOIN legacy_parsed lp
        ON dm.work_id = lp.work_id
        AND dm.raw_last = lp.legacy_last
        AND dm.raw_first = lp.legacy_first
        AND lp.rn_parsed = 1
    WHERE dm.author_id != lp.author_id
      AND NOT EXISTS (
          SELECT 1 FROM exact_match em
          WHERE em.work_id = dm.work_id AND em.author_sequence = dm.author_sequence
      )
),
all_matches AS (
    SELECT * FROM exact_match
    UNION ALL
    SELECT * FROM parsed_match
)
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT work_id) AS work_count
FROM all_matches

### Cell 2: Identify affected work_ids

Works where at least one position has a parsed last name mismatch AND a legacy join (exact name OR parsed name fallback) gives a different author_id.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.author_sequence_repair_work_ids AS
WITH display_mismatch AS (
    SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name,
           pn_raw.parsed_name.last AS raw_last, pn_raw.parsed_name.first AS raw_first
    FROM openalex.works.work_authors wa
    INNER JOIN openalex.authors.openalex_authors oa ON wa.author_id = oa.id
    INNER JOIN openalex.authors.author_names pn_raw ON TRIM(wa.raw_author_name) = pn_raw.raw_author_name
    INNER JOIN openalex.authors.author_names pn_display ON TRIM(oa.display_name) = pn_display.raw_author_name
    WHERE wa.work_id <= 7000000000
      AND wa.author_id IS NOT NULL
      AND pn_raw.parsed_name.last IS NOT NULL AND pn_raw.parsed_name.last != ''
      AND pn_display.parsed_name.last IS NOT NULL AND pn_display.parsed_name.last != ''
      AND pn_raw.parsed_name.last != pn_display.parsed_name.last
      AND pn_display.parsed_name.last NOT IN (pn_raw.parsed_name.first, pn_raw.parsed_name.middle, pn_raw.parsed_name.last)
      AND pn_raw.parsed_name.last NOT IN (pn_display.parsed_name.first, pn_display.parsed_name.middle, pn_display.parsed_name.last)
      AND EXISTS (
          SELECT 1 FROM openalex.works.work_authors wa2
          WHERE wa2.work_id = wa.work_id AND wa2.author_sequence > 0
      )
),
-- Legacy enriched with parsed names
legacy_parsed AS (
    SELECT lwa.work_id, LOWER(TRIM(lwa.raw_author_name)) AS name_norm, lwa.author_id, lwa.author_sequence,
           pn.parsed_name.last AS legacy_last, pn.parsed_name.first AS legacy_first,
           ROW_NUMBER() OVER (PARTITION BY lwa.work_id, LOWER(TRIM(lwa.raw_author_name)) ORDER BY lwa.author_sequence) AS rn_exact,
           ROW_NUMBER() OVER (PARTITION BY lwa.work_id, pn.parsed_name.last, pn.parsed_name.first ORDER BY lwa.author_sequence) AS rn_parsed
    FROM openalex.works_legacy.work_authors lwa
    INNER JOIN openalex.authors.author_names pn ON TRIM(lwa.raw_author_name) = pn.raw_author_name
    WHERE lwa.author_id IS NOT NULL
),
-- Pass 1: exact name match
exact_match AS (
    SELECT dm.work_id, dm.author_sequence, dm.author_id, lp.author_id AS legacy_author_id
    FROM display_mismatch dm
    INNER JOIN legacy_parsed lp
        ON dm.work_id = lp.work_id
        AND LOWER(TRIM(dm.raw_author_name)) = lp.name_norm
        AND lp.rn_exact = 1
    WHERE dm.author_id != lp.author_id
),
-- Pass 2: parsed name fallback for rows not matched in Pass 1
parsed_match AS (
    SELECT dm.work_id, dm.author_sequence, dm.author_id, lp.author_id AS legacy_author_id
    FROM display_mismatch dm
    INNER JOIN legacy_parsed lp
        ON dm.work_id = lp.work_id
        AND dm.raw_last = lp.legacy_last
        AND dm.raw_first = lp.legacy_first
        AND lp.rn_parsed = 1
    WHERE dm.author_id != lp.author_id
      AND NOT EXISTS (
          SELECT 1 FROM exact_match em
          WHERE em.work_id = dm.work_id AND em.author_sequence = dm.author_sequence
      )
),
all_matches AS (
    SELECT work_id FROM exact_match
    UNION
    SELECT work_id FROM parsed_match
)
SELECT DISTINCT work_id FROM all_matches

### Cell 3: Build repair table — Pass 1 (exact name match)

For affected works only, join `work_authors` to backfill on exact normalized `raw_author_name`. Uses `ROW_NUMBER` tiebreaker for duplicate names on the same paper.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.author_sequence_repair AS
WITH legacy_wa AS (
    SELECT
        work_id,
        raw_author_name AS legacy_name,
        LOWER(TRIM(raw_author_name)) AS legacy_name_norm,
        author_id AS legacy_author_id,
        author_sequence AS legacy_sequence
    FROM openalex.works_legacy.work_authors
    WHERE author_id IS NOT NULL
      AND work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
),
current_wa AS (
    SELECT
        work_id,
        author_sequence,
        author_id,
        raw_author_name,
        LOWER(TRIM(raw_author_name)) AS current_name_norm
    FROM openalex.works.work_authors
    WHERE work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
),
name_matched AS (
    SELECT
        cwa.work_id,
        cwa.author_sequence,
        cwa.author_id AS current_author_id,
        cwa.raw_author_name,
        lwa.legacy_author_id,
        lwa.legacy_sequence,
        ROW_NUMBER() OVER (
            PARTITION BY cwa.work_id, cwa.current_name_norm
            ORDER BY cwa.author_sequence
        ) AS current_rank,
        ROW_NUMBER() OVER (
            PARTITION BY lwa.work_id, lwa.legacy_name_norm
            ORDER BY lwa.legacy_sequence
        ) AS legacy_rank
    FROM current_wa cwa
    INNER JOIN legacy_wa lwa
        ON cwa.work_id = lwa.work_id
        AND cwa.current_name_norm = lwa.legacy_name_norm
)
SELECT
    work_id,
    author_sequence,
    raw_author_name,
    current_author_id,
    legacy_author_id AS backfill_author_id,
    'EXACT' AS match_type,
    CASE
        WHEN current_author_id = legacy_author_id THEN 'ALREADY_CORRECT'
        WHEN current_author_id IS NULL THEN 'WAS_NULL'
        ELSE 'WRONG_AUTHOR_ID'
    END AS repair_status
FROM name_matched
WHERE current_rank = legacy_rank

### Cell 3b: Build repair table — Pass 2 (parsed name fallback)

For affected work_authors rows not matched in Pass 1, join via `author_names` on `(work_id, parsed last, parsed first)`. This catches provenance-driven name variations like accent differences, period handling, and initial formatting.

In [0]:
INSERT INTO openalex.authors.author_sequence_repair
WITH legacy_wa AS (
    SELECT
        work_id,
        raw_author_name AS legacy_name,
        author_id AS legacy_author_id,
        author_sequence AS legacy_sequence
    FROM openalex.works_legacy.work_authors
    WHERE author_id IS NOT NULL
      AND work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
),
-- Legacy names enriched with parsed components
legacy_parsed AS (
    SELECT
        lwa.*,
        pn.parsed_name.last AS legacy_last,
        pn.parsed_name.first AS legacy_first
    FROM legacy_wa lwa
    INNER JOIN openalex.authors.author_names pn
        ON TRIM(lwa.legacy_name) = pn.raw_author_name
    WHERE pn.parsed_name.last IS NOT NULL AND pn.parsed_name.last != ''
),
-- Unmatched work_authors rows from Pass 1
unmatched_wa AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name
    FROM openalex.works.work_authors wa
    LEFT JOIN openalex.authors.author_sequence_repair r
        ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
    WHERE wa.work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
      AND r.work_id IS NULL
),
-- Unmatched rows enriched with parsed components
unmatched_parsed AS (
    SELECT
        uwa.*,
        pn.parsed_name.last AS current_last,
        pn.parsed_name.first AS current_first
    FROM unmatched_wa uwa
    INNER JOIN openalex.authors.author_names pn
        ON TRIM(uwa.raw_author_name) = pn.raw_author_name
    WHERE pn.parsed_name.last IS NOT NULL AND pn.parsed_name.last != ''
),
-- Join on parsed name components
parsed_matched AS (
    SELECT
        up.work_id,
        up.author_sequence,
        up.author_id AS current_author_id,
        up.raw_author_name,
        lp.legacy_author_id,
        lp.legacy_sequence,
        ROW_NUMBER() OVER (
            PARTITION BY up.work_id, up.current_last, up.current_first
            ORDER BY up.author_sequence
        ) AS current_rank,
        ROW_NUMBER() OVER (
            PARTITION BY lp.work_id, lp.legacy_last, lp.legacy_first
            ORDER BY lp.legacy_sequence
        ) AS legacy_rank
    FROM unmatched_parsed up
    INNER JOIN legacy_parsed lp
        ON up.work_id = lp.work_id
        AND up.current_last = lp.legacy_last
        AND up.current_first = lp.legacy_first
)
SELECT
    work_id,
    author_sequence,
    raw_author_name,
    current_author_id,
    legacy_author_id AS backfill_author_id,
    'PARSED' AS match_type,
    CASE
        WHEN current_author_id = legacy_author_id THEN 'ALREADY_CORRECT'
        WHEN current_author_id IS NULL THEN 'WAS_NULL'
        ELSE 'WRONG_AUTHOR_ID'
    END AS repair_status
FROM parsed_matched
WHERE current_rank = legacy_rank

### Cell 3c: Diagnostic — Analyze repair coverage

Review repair_status and match_type distribution, plus count of unmatched rows still in affected works.

In [0]:
SELECT match_type, repair_status, COUNT(*) AS row_count, COUNT(DISTINCT work_id) AS work_count
FROM openalex.authors.author_sequence_repair
GROUP BY match_type, repair_status

UNION ALL

SELECT
    'N/A' AS match_type,
    'UNMATCHED' AS repair_status,
    COUNT(*) AS row_count,
    COUNT(DISTINCT wa.work_id) AS work_count
FROM openalex.works.work_authors wa
LEFT JOIN openalex.authors.author_sequence_repair r
    ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
WHERE wa.work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
  AND r.work_id IS NULL

### Cell 4: Apply repair via MERGE

Restore correct author_ids from backfill ground truth.

In [0]:
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT work_id, author_sequence, backfill_author_id
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY work_id, author_sequence
                ORDER BY
                    CASE match_type WHEN 'EXACT' THEN 1 ELSE 2 END,
                    CASE repair_status WHEN 'WRONG_AUTHOR_ID' THEN 1 WHEN 'WAS_NULL' THEN 2 ELSE 3 END
            ) AS dedup_rank
        FROM openalex.authors.author_sequence_repair
        WHERE repair_status IN ('WRONG_AUTHOR_ID', 'WAS_NULL')
    )
    WHERE dedup_rank = 1
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN UPDATE SET
    target.author_id = source.backfill_author_id,
    target.updated_at = current_timestamp()

### Cell 5: NULL unmatched rows with duplicate author_id on same work

For repair-scoped rows not covered by the repair table, NULL author_ids where the same author_id appears at multiple positions on the same work. This is strong evidence of a wrong assignment from the sequence bug.

In [0]:
MERGE INTO openalex.works.work_authors AS target
USING (
    SELECT wa.work_id, wa.author_sequence
    FROM openalex.works.work_authors wa
    INNER JOIN (
        -- Find duplicate author_ids within repair-scoped works
        SELECT work_id, author_id
        FROM openalex.works.work_authors
        WHERE work_id IN (SELECT work_id FROM openalex.authors.author_sequence_repair_work_ids)
          AND author_id IS NOT NULL
        GROUP BY work_id, author_id
        HAVING COUNT(*) > 1
    ) dupes ON wa.work_id = dupes.work_id AND wa.author_id = dupes.author_id
    -- Only target rows NOT already handled by the repair table
    LEFT JOIN openalex.authors.author_sequence_repair r
        ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
    WHERE r.work_id IS NULL
) AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED THEN UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 6: Queue affected works for UpdateWorkAuthorships

Insert repaired work_ids into `curated_work_ids_pending_sync` so the next E2E run propagates fixes.

In [0]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync (work_id, curated_ras, added_datetime)
SELECT DISTINCT
    work_id,
    'author_sequence_repair' AS curated_ras,
    current_timestamp() AS added_datetime
FROM (
    -- Works repaired from backfill (Cell 4)
    SELECT work_id
    FROM openalex.authors.author_sequence_repair
    WHERE repair_status IN ('WRONG_AUTHOR_ID', 'WAS_NULL')

    UNION

    -- Works with duplicate author_ids NULLed (Cell 5)
    SELECT work_id
    FROM openalex.authors.author_sequence_repair_work_ids
    WHERE work_id IN (
        SELECT work_id FROM openalex.works.work_authors
        WHERE author_id IS NULL
          AND updated_at >= current_timestamp() - INTERVAL 2 HOURS
    )
) affected
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.institutions.curated_work_ids_pending_sync p
    WHERE p.work_id = affected.work_id
)

### Cell 7: Final verification

Sample repaired rows to confirm author_ids now match backfill.

In [0]:
SELECT
    wa.work_id,
    wa.author_sequence,
    wa.author_id AS new_author_id,
    wa.raw_author_name,
    r.current_author_id AS old_author_id,
    r.backfill_author_id,
    r.repair_status
FROM openalex.works.work_authors wa
INNER JOIN openalex.authors.author_sequence_repair r
    ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
WHERE r.repair_status = 'WRONG_AUTHOR_ID'
LIMIT 100

---

## Phase 4: Broad Mismatch Repair

Cells 1-7 above fixed 834K works using legacy `work_authors` as ground truth. This section addresses the **remaining ~2.2M mismatches** detected by a broad scan of `work_authors` + `openalex_authors` where `raw_author_name` and entity `display_name` have no resemblance.

### Detection Method
1. **SOUNDEX zero overlap** — tokenize both names, SOUNDEX each token, check for zero intersection
2. **Script filter** — both names must have Latin tokens >=3 chars (removes transliteration FPs)
3. **Institutional filter** — raw 2-6 tokens, display 2-5 tokens
4. **Diacritics filter** — relative Levenshtein: `min(lev/maxlen)` across all token pairs must be >0.34

Estimated true positive rate: **~92%**

### Fix Strategy
NULL wrong `author_id` values so `MatchAuthors` can re-match after `CreateAuthors` cleans poisoned profiles. Unlike Cells 1-7 (which replaced with a known-correct author_id from legacy), we don't have a correction — we just know the current assignment is wrong.

### Prerequisites
- Run **after** Cells 1-7 (so already-repaired rows are excluded)
- Run on X-Large SQL warehouse

### Cell 8: Diagnostic — Count broad mismatches

Count `work_authors` rows where `raw_author_name` and entity `display_name` fail all similarity checks (SOUNDEX, Levenshtein, script).

In [0]:
WITH base AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name,
        oa.display_name,
        FILTER(SPLIT(LOWER(TRANSLATE(TRIM(wa.raw_author_name), '.,;-', '    ')), ' '), x -> LENGTH(TRIM(x)) > 2) AS raw_toks,
        FILTER(SPLIT(LOWER(TRANSLATE(TRIM(oa.display_name), '.,;-', '    ')), ' '), x -> LENGTH(TRIM(x)) > 2) AS disp_toks
    FROM openalex.works.work_authors wa
    INNER JOIN openalex.authors.openalex_authors oa ON wa.author_id = oa.id
    WHERE wa.author_id IS NOT NULL
      AND wa.raw_author_name IS NOT NULL
      AND LENGTH(wa.raw_author_name) > 3
      AND LENGTH(oa.display_name) > 5
      AND SIZE(SPLIT(wa.raw_author_name, ' ')) BETWEEN 2 AND 6
      AND SIZE(SPLIT(oa.display_name, ' ')) BETWEEN 2 AND 5
      AND LOWER(wa.raw_author_name) NOT LIKE '%anonymous%'
      AND LOWER(wa.raw_author_name) NOT LIKE '%ai generated%'
      AND LOWER(wa.raw_author_name) NOT LIKE '%withheld%'
      AND LOWER(wa.raw_author_name) NOT LIKE '%usdoe%'
      -- SOUNDEX zero overlap
      AND SIZE(ARRAY_INTERSECT(
          TRANSFORM(SPLIT(LOWER(TRANSLATE(TRIM(wa.raw_author_name), '.,', '  ')), ' '), x -> SOUNDEX(TRIM(x))),
          TRANSFORM(SPLIT(LOWER(TRANSLATE(TRIM(oa.display_name), '.,', '  ')), ' '), x -> SOUNDEX(TRIM(x)))
      )) = 0
      -- Both names have Latin tokens >=3 chars
      AND SIZE(FILTER(REGEXP_EXTRACT_ALL(LOWER(wa.raw_author_name), '([a-z]{3,})', 0), x -> true)) > 0
      AND SIZE(FILTER(REGEXP_EXTRACT_ALL(LOWER(oa.display_name), '([a-z]{3,})', 0), x -> true)) > 0
)
SELECT
    CASE WHEN work_id > 7000000000 THEN '>7B' ELSE '<=7B' END AS era,
    COUNT(*) AS mismatch_rows,
    COUNT(DISTINCT work_id) AS mismatch_works
FROM base
WHERE SIZE(raw_toks) > 0 AND SIZE(disp_toks) > 0
  -- Relative Levenshtein: no token pair is close enough to be diacritics
  AND AGGREGATE(
      FLATTEN(TRANSFORM(raw_toks, rt ->
          TRANSFORM(disp_toks, dt ->
              CAST(LEVENSHTEIN(TRIM(rt), TRIM(dt)) AS DOUBLE) / CAST(GREATEST(LENGTH(TRIM(rt)), LENGTH(TRIM(dt)), 1) AS DOUBLE)
          )
      )),
      CAST(999.0 AS DOUBLE), (acc, x) -> LEAST(acc, x)
  ) > 0.34
GROUP BY 1

### Cell 9: Build broad mismatch table

Materialize the detected mismatches into `author_sequence_broad_repair` for inspection and repair.

In [0]:
CREATE OR REPLACE TABLE openalex.authors.author_sequence_broad_repair AS
WITH base AS (
    SELECT
        wa.work_id,
        wa.author_sequence,
        wa.author_id,
        wa.raw_author_name,
        oa.display_name,
        FILTER(SPLIT(LOWER(TRANSLATE(TRIM(wa.raw_author_name), '.,;-', '    ')), ' '), x -> LENGTH(TRIM(x)) > 2) AS raw_toks,
        FILTER(SPLIT(LOWER(TRANSLATE(TRIM(oa.display_name), '.,;-', '    ')), ' '), x -> LENGTH(TRIM(x)) > 2) AS disp_toks
    FROM openalex.works.work_authors wa
    INNER JOIN openalex.authors.openalex_authors oa ON wa.author_id = oa.id
    WHERE wa.author_id IS NOT NULL
      AND wa.raw_author_name IS NOT NULL
      AND LENGTH(wa.raw_author_name) > 3
      AND LENGTH(oa.display_name) > 5
      AND SIZE(SPLIT(wa.raw_author_name, ' ')) BETWEEN 2 AND 6
      AND SIZE(SPLIT(oa.display_name, ' ')) BETWEEN 2 AND 5
      AND LOWER(wa.raw_author_name) NOT LIKE '%anonymous%'
      AND LOWER(wa.raw_author_name) NOT LIKE '%ai generated%'
      AND LOWER(wa.raw_author_name) NOT LIKE '%withheld%'
      AND LOWER(wa.raw_author_name) NOT LIKE '%usdoe%'
      -- SOUNDEX zero overlap
      AND SIZE(ARRAY_INTERSECT(
          TRANSFORM(SPLIT(LOWER(TRANSLATE(TRIM(wa.raw_author_name), '.,', '  ')), ' '), x -> SOUNDEX(TRIM(x))),
          TRANSFORM(SPLIT(LOWER(TRANSLATE(TRIM(oa.display_name), '.,', '  ')), ' '), x -> SOUNDEX(TRIM(x)))
      )) = 0
      -- Both names have Latin tokens >=3 chars
      AND SIZE(FILTER(REGEXP_EXTRACT_ALL(LOWER(wa.raw_author_name), '([a-z]{3,})', 0), x -> true)) > 0
      AND SIZE(FILTER(REGEXP_EXTRACT_ALL(LOWER(oa.display_name), '([a-z]{3,})', 0), x -> true)) > 0
)
SELECT
    work_id,
    author_sequence,
    author_id,
    raw_author_name,
    display_name AS entity_display_name
FROM base
WHERE SIZE(raw_toks) > 0 AND SIZE(disp_toks) > 0
  AND AGGREGATE(
      FLATTEN(TRANSFORM(raw_toks, rt ->
          TRANSFORM(disp_toks, dt ->
              CAST(LEVENSHTEIN(TRIM(rt), TRIM(dt)) AS DOUBLE) / CAST(GREATEST(LENGTH(TRIM(rt)), LENGTH(TRIM(dt)), 1) AS DOUBLE)
          )
      )),
      CAST(999.0 AS DOUBLE), (acc, x) -> LEAST(acc, x)
  ) > 0.34

### Cell 9b: Diagnostic — Inspect broad mismatch table

Sample and count the detected mismatches before applying the NULL repair.

In [0]:
SELECT
    CASE WHEN work_id > 7000000000 THEN '>7B' ELSE '<=7B' END AS era,
    COUNT(*) AS rows,
    COUNT(DISTINCT work_id) AS works,
    COUNT(DISTINCT author_id) AS authors
FROM openalex.authors.author_sequence_broad_repair
GROUP BY 1

UNION ALL

SELECT 'SAMPLE' AS era, NULL, NULL, NULL
FROM openalex.authors.author_sequence_broad_repair
LIMIT 0

### Cell 9c: Sample broad mismatches for manual review

Spot-check a random sample before proceeding with the NULL pass.

In [0]:
SELECT work_id, author_sequence, raw_author_name, entity_display_name, author_id
FROM openalex.authors.author_sequence_broad_repair
ORDER BY RAND(42)
LIMIT 50

### Cell 10: Apply broad repair — NULL wrong author_ids

NULL the `author_id` for all detected mismatches. These will be re-matched by `MatchAuthors` after `CreateAuthors` cleans the poisoned profiles.

In [0]:
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.author_sequence_broad_repair AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
   AND target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 11: Queue broad repair works for reprocessing

Insert affected work_ids into `curated_work_ids_pending_sync` so `UpdateWorkAuthorships` propagates the NULLs, and `MatchAuthors` re-matches after `CreateAuthors` runs.

In [0]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync (work_id, curated_ras, added_datetime)
SELECT DISTINCT
    work_id,
    'broad_mismatch_repair' AS curated_ras,
    current_timestamp() AS added_datetime
FROM openalex.authors.author_sequence_broad_repair
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.institutions.curated_work_ids_pending_sync p
    WHERE p.work_id = openalex.authors.author_sequence_broad_repair.work_id
)

### Cell 12: Verify broad repair

Confirm that the NULLed rows are reflected in `work_authors`.

In [0]:
SELECT
    CASE WHEN wa.author_id IS NULL THEN 'NULLED' ELSE 'NOT_NULLED' END AS status,
    COUNT(*) AS row_count,
    COUNT(DISTINCT br.work_id) AS work_count
FROM openalex.authors.author_sequence_broad_repair br
INNER JOIN openalex.works.work_authors wa
    ON br.work_id = wa.work_id AND br.author_sequence = wa.author_sequence
GROUP BY 1